# DTW Benchmarking: CPU vs GPU

This notebook compares the CPU (tslearn + numpy) and GPU (CUDA) backends for DTW computation.

**What we benchmark:**
1. Correctness — do CPU and GPU produce the same results?
2. `dtw_distance` — single-pair timing across sequence lengths
3. `dtw_pairwise` — batch timing across number of sequences
4. Open boundary conditions (`open_start`, `open_end`)

**Note:** On a CPU-only machine, GPU benchmarks are skipped and only CPU results are shown.

In [ ]:
import time
import numpy as np
import pandas as pd
from baleen._cuda_dtw import dtw_distance, dtw_pairwise, backend, CUDA_AVAILABLE

print(f"Backend: {backend()}")
print(f"CUDA available: {CUDA_AVAILABLE}")

## 1. Correctness Verification

If GPU is available, verify that CPU and GPU produce identical results (within floating-point tolerance).

In [ ]:
rng = np.random.default_rng(42)

if CUDA_AVAILABLE:
    print("=== Correctness: dtw_distance ===")
    for length in [10, 50, 100, 500]:
        seq1 = rng.standard_normal(length).astype(np.float32)
        seq2 = rng.standard_normal(length).astype(np.float32)
        d_cpu = dtw_distance(seq1, seq2, use_cuda=False)
        d_gpu = dtw_distance(seq1, seq2, use_cuda=True)
        diff = abs(d_cpu - d_gpu)
        status = "✅" if diff < 1e-4 else "❌"
        print(f"  len={length:>4d}  CPU={d_cpu:.6f}  GPU={d_gpu:.6f}  diff={diff:.2e}  {status}")

    print("\n=== Correctness: dtw_distance (open_end=True) ===")
    for length in [10, 50, 100, 500]:
        seq1 = rng.standard_normal(length).astype(np.float32)
        seq2 = rng.standard_normal(length + length // 2).astype(np.float32)
        d_cpu = dtw_distance(seq1, seq2, use_open_end=True, use_cuda=False)
        d_gpu = dtw_distance(seq1, seq2, use_open_end=True, use_cuda=True)
        diff = abs(d_cpu - d_gpu)
        status = "✅" if diff < 1e-4 else "❌"
        print(f"  len={length:>4d}  CPU={d_cpu:.6f}  GPU={d_gpu:.6f}  diff={diff:.2e}  {status}")

    print("\n=== Correctness: dtw_distance (open_start=True) ===")
    for length in [10, 50, 100, 500]:
        seq1 = rng.standard_normal(length).astype(np.float32)
        seq2 = rng.standard_normal(length + length // 2).astype(np.float32)
        d_cpu = dtw_distance(seq1, seq2, use_open_start=True, use_cuda=False)
        d_gpu = dtw_distance(seq1, seq2, use_open_start=True, use_cuda=True)
        diff = abs(d_cpu - d_gpu)
        status = "✅" if diff < 1e-4 else "❌"
        print(f"  len={length:>4d}  CPU={d_cpu:.6f}  GPU={d_gpu:.6f}  diff={diff:.2e}  {status}")

    print("\n=== Correctness: dtw_pairwise ===")
    for n_seq in [5, 20, 50]:
        seqs = rng.standard_normal((n_seq, 100)).astype(np.float32)
        m_cpu = dtw_pairwise(seqs, use_cuda=False)
        m_gpu = dtw_pairwise(seqs, use_cuda=True)
        max_diff = np.max(np.abs(m_cpu - m_gpu))
        status = "✅" if max_diff < 1e-3 else "❌"
        print(f"  n={n_seq:>3d}  max_diff={max_diff:.2e}  {status}")
else:
    print("GPU not available — skipping correctness comparison.")
    print("CPU-only results will be shown in the benchmarks below.")

## 2. Single-pair `dtw_distance` Benchmark

Measure time for a single `dtw_distance` call across increasing sequence lengths.

In [ ]:
def bench_dtw_distance(seq_lengths, n_repeats=5):
    """Benchmark dtw_distance across sequence lengths."""
    rng = np.random.default_rng(123)
    rows = []

    for length in seq_lengths:
        seq1 = rng.standard_normal(length).astype(np.float32)
        seq2 = rng.standard_normal(length).astype(np.float32)

        # CPU (tslearn)
        times_cpu = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            dtw_distance(seq1, seq2, use_cuda=False)
            times_cpu.append(time.perf_counter() - t0)
        cpu_ms = np.median(times_cpu) * 1000

        row = {"seq_length": length, "cpu_ms": cpu_ms}

        # GPU
        if CUDA_AVAILABLE:
            # warmup
            dtw_distance(seq1, seq2, use_cuda=True)
            times_gpu = []
            for _ in range(n_repeats):
                t0 = time.perf_counter()
                dtw_distance(seq1, seq2, use_cuda=True)
                times_gpu.append(time.perf_counter() - t0)
            gpu_ms = np.median(times_gpu) * 1000
            row["gpu_ms"] = gpu_ms
            row["speedup"] = cpu_ms / gpu_ms if gpu_ms > 0 else float("inf")

        rows.append(row)

    return pd.DataFrame(rows)


seq_lengths = [50, 100, 200, 500, 1000, 2000, 5000]
df_single = bench_dtw_distance(seq_lengths)
print("=== dtw_distance benchmark (median, ms) ===")
print(df_single.to_string(index=False, float_format="%.3f"))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: absolute time
ax = axes[0]
ax.plot(df_single["seq_length"], df_single["cpu_ms"], "o-", label="CPU (tslearn)", color="tab:blue")
if "gpu_ms" in df_single.columns:
    ax.plot(df_single["seq_length"], df_single["gpu_ms"], "s-", label="GPU (CUDA)", color="tab:orange")
ax.set_xlabel("Sequence Length")
ax.set_ylabel("Time (ms)")
ax.set_title("dtw_distance: Single Pair")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: speedup
ax = axes[1]
if "speedup" in df_single.columns:
    ax.bar(df_single["seq_length"].astype(str), df_single["speedup"], color="tab:green")
    ax.set_xlabel("Sequence Length")
    ax.set_ylabel("Speedup (CPU / GPU)")
    ax.set_title("GPU Speedup")
    ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5)
    ax.grid(True, alpha=0.3, axis="y")
else:
    ax.text(0.5, 0.5, "GPU not available", ha="center", va="center", transform=ax.transAxes, fontsize=14)
    ax.set_title("GPU Speedup (N/A)")

plt.tight_layout()
plt.show()

## 3. Pairwise `dtw_pairwise` Benchmark

Measure time for computing the full pairwise distance matrix across increasing batch sizes (fixed sequence length = 100).

In [ ]:
def bench_dtw_pairwise(n_seqs_list, seq_length=100, n_repeats=3):
    """Benchmark dtw_pairwise across batch sizes."""
    rng = np.random.default_rng(456)
    rows = []

    for n_seq in n_seqs_list:
        seqs = rng.standard_normal((n_seq, seq_length)).astype(np.float32)
        n_pairs = n_seq * (n_seq - 1) // 2

        # CPU
        times_cpu = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            dtw_pairwise(seqs, use_cuda=False)
            times_cpu.append(time.perf_counter() - t0)
        cpu_s = np.median(times_cpu)
        cpu_pairs_sec = n_pairs / cpu_s if cpu_s > 0 else 0

        row = {
            "n_sequences": n_seq,
            "n_pairs": n_pairs,
            "cpu_sec": cpu_s,
            "cpu_pairs/sec": cpu_pairs_sec,
        }

        # GPU
        if CUDA_AVAILABLE:
            # warmup
            dtw_pairwise(seqs, use_cuda=True)
            times_gpu = []
            for _ in range(n_repeats):
                t0 = time.perf_counter()
                dtw_pairwise(seqs, use_cuda=True)
                times_gpu.append(time.perf_counter() - t0)
            gpu_s = np.median(times_gpu)
            gpu_pairs_sec = n_pairs / gpu_s if gpu_s > 0 else 0
            row["gpu_sec"] = gpu_s
            row["gpu_pairs/sec"] = gpu_pairs_sec
            row["speedup"] = cpu_s / gpu_s if gpu_s > 0 else float("inf")

        rows.append(row)
        print(f"  n_seq={n_seq:>4d}  pairs={n_pairs:>7d}  done")

    return pd.DataFrame(rows)


# Adjust sizes based on whether GPU is available
# CPU-only: keep small to avoid long waits
if CUDA_AVAILABLE:
    n_seqs_list = [10, 20, 50, 100, 200, 500, 1000]
else:
    n_seqs_list = [10, 20, 50, 100, 200]

print(f"Benchmarking pairwise DTW (seq_length=100)...")
df_pairwise = bench_dtw_pairwise(n_seqs_list)
print("\n=== dtw_pairwise benchmark ===")
print(df_pairwise.to_string(index=False, float_format="%.4f"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: time vs n_sequences
ax = axes[0]
ax.plot(df_pairwise["n_sequences"], df_pairwise["cpu_sec"], "o-", label="CPU (tslearn)", color="tab:blue")
if "gpu_sec" in df_pairwise.columns:
    ax.plot(df_pairwise["n_sequences"], df_pairwise["gpu_sec"], "s-", label="GPU (CUDA)", color="tab:orange")
ax.set_xlabel("Number of Sequences")
ax.set_ylabel("Time (seconds)")
ax.set_title("dtw_pairwise: Total Time")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: throughput (pairs/sec)
ax = axes[1]
ax.plot(df_pairwise["n_sequences"], df_pairwise["cpu_pairs/sec"], "o-", label="CPU", color="tab:blue")
if "gpu_pairs/sec" in df_pairwise.columns:
    ax.plot(df_pairwise["n_sequences"], df_pairwise["gpu_pairs/sec"], "s-", label="GPU", color="tab:orange")
ax.set_xlabel("Number of Sequences")
ax.set_ylabel("Throughput (pairs/sec)")
ax.set_title("dtw_pairwise: Throughput")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Open Boundary Benchmark

Compare standard DTW vs open-boundary DTW timing on CPU (and GPU if available).  
CPU standard DTW uses tslearn; CPU open-boundary uses the numpy DP fallback.

In [ ]:
def bench_open_boundary(seq_lengths, n_repeats=5):
    """Benchmark standard vs open_end DTW."""
    rng = np.random.default_rng(789)
    rows = []

    for length in seq_lengths:
        seq1 = rng.standard_normal(length).astype(np.float32)
        seq2 = rng.standard_normal(int(length * 1.5)).astype(np.float32)

        # CPU standard
        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            dtw_distance(seq1, seq2, use_cuda=False)
            times.append(time.perf_counter() - t0)
        cpu_std_ms = np.median(times) * 1000

        # CPU open_end
        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            dtw_distance(seq1, seq2, use_open_end=True, use_cuda=False)
            times.append(time.perf_counter() - t0)
        cpu_open_ms = np.median(times) * 1000

        # CPU open_start
        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            dtw_distance(seq1, seq2, use_open_start=True, use_cuda=False)
            times.append(time.perf_counter() - t0)
        cpu_open_start_ms = np.median(times) * 1000

        row = {
            "seq_length": length,
            "cpu_standard_ms": cpu_std_ms,
            "cpu_open_end_ms": cpu_open_ms,
            "cpu_open_start_ms": cpu_open_start_ms,
            "open_end_overhead": cpu_open_ms / cpu_std_ms if cpu_std_ms > 0 else 0,
        }

        if CUDA_AVAILABLE:
            # GPU warmup
            dtw_distance(seq1, seq2, use_cuda=True)

            # GPU standard
            times = []
            for _ in range(n_repeats):
                t0 = time.perf_counter()
                dtw_distance(seq1, seq2, use_cuda=True)
                times.append(time.perf_counter() - t0)
            row["gpu_standard_ms"] = np.median(times) * 1000

            # GPU open_end
            times = []
            for _ in range(n_repeats):
                t0 = time.perf_counter()
                dtw_distance(seq1, seq2, use_open_end=True, use_cuda=True)
                times.append(time.perf_counter() - t0)
            row["gpu_open_end_ms"] = np.median(times) * 1000

            # GPU open_start
            times = []
            for _ in range(n_repeats):
                t0 = time.perf_counter()
                dtw_distance(seq1, seq2, use_open_start=True, use_cuda=True)
                times.append(time.perf_counter() - t0)
            row["gpu_open_start_ms"] = np.median(times) * 1000

        rows.append(row)

    return pd.DataFrame(rows)


open_lengths = [50, 100, 200, 500, 1000]
df_open = bench_open_boundary(open_lengths)
print("=== Open Boundary benchmark (median, ms) ===")
print(df_open.to_string(index=False, float_format="%.3f"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: CPU standard vs open_end vs open_start
ax = axes[0]
ax.plot(df_open["seq_length"], df_open["cpu_standard_ms"], "o-", label="CPU standard (tslearn)", color="tab:blue")
ax.plot(df_open["seq_length"], df_open["cpu_open_end_ms"], "s-", label="CPU open_end (numpy)", color="tab:red")
ax.plot(df_open["seq_length"], df_open["cpu_open_start_ms"], "^-", label="CPU open_start (numpy)", color="tab:purple")
if "gpu_standard_ms" in df_open.columns:
    ax.plot(df_open["seq_length"], df_open["gpu_standard_ms"], "D--", label="GPU standard", color="tab:orange")
    ax.plot(df_open["seq_length"], df_open["gpu_open_end_ms"], "v--", label="GPU open_end", color="tab:green")
ax.set_xlabel("Sequence Length")
ax.set_ylabel("Time (ms)")
ax.set_title("Standard vs Open Boundary")
ax.set_yscale("log")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: overhead ratio (open_end_time / standard_time) on CPU
ax = axes[1]
ax.bar(df_open["seq_length"].astype(str), df_open["open_end_overhead"], color="tab:red", alpha=0.7)
ax.set_xlabel("Sequence Length")
ax.set_ylabel("Ratio (open_end / standard)")
ax.set_title("CPU open_end overhead vs tslearn")
ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 5. Pairwise Open Boundary Benchmark

Compare pairwise standard (tslearn `cdist_dtw`) vs pairwise open_end (numpy loop) on CPU.

In [ ]:
def bench_pairwise_open(n_seqs_list, seq_length=50, n_repeats=3):
    """Benchmark pairwise: standard (tslearn) vs open_end (numpy loop)."""
    rng = np.random.default_rng(321)
    rows = []

    for n_seq in n_seqs_list:
        seqs = rng.standard_normal((n_seq, seq_length)).astype(np.float32)
        n_pairs = n_seq * (n_seq - 1) // 2

        # CPU standard (tslearn cdist_dtw)
        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            dtw_pairwise(seqs, use_cuda=False)
            times.append(time.perf_counter() - t0)
        cpu_std_s = np.median(times)

        # CPU open_end (numpy loop)
        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            dtw_pairwise(seqs, use_open_end=True, use_cuda=False)
            times.append(time.perf_counter() - t0)
        cpu_open_s = np.median(times)

        row = {
            "n_sequences": n_seq,
            "n_pairs": n_pairs,
            "cpu_standard_sec": cpu_std_s,
            "cpu_open_end_sec": cpu_open_s,
            "open_end_overhead": cpu_open_s / cpu_std_s if cpu_std_s > 0 else 0,
        }

        if CUDA_AVAILABLE:
            dtw_pairwise(seqs, use_cuda=True)  # warmup
            times = []
            for _ in range(n_repeats):
                t0 = time.perf_counter()
                dtw_pairwise(seqs, use_open_end=True, use_cuda=True)
                times.append(time.perf_counter() - t0)
            row["gpu_open_end_sec"] = np.median(times)

        rows.append(row)
        print(f"  n_seq={n_seq:>3d}  pairs={n_pairs:>5d}  done")

    return pd.DataFrame(rows)


# Keep small — numpy loop is O(n^2 * L^2)
pairwise_open_sizes = [5, 10, 20, 50]
print("Benchmarking pairwise open boundary (seq_length=50)...")
df_pw_open = bench_pairwise_open(pairwise_open_sizes)
print("\n=== Pairwise open boundary benchmark ===")
print(df_pw_open.to_string(index=False, float_format="%.4f"))

## 6. Summary

In [ ]:
print("=" * 60)
print("BENCHMARK SUMMARY")
print("=" * 60)
print(f"Backend: {backend()}")
print(f"CUDA available: {CUDA_AVAILABLE}")
print()

print("--- dtw_distance (single pair) ---")
print(df_single[["seq_length", "cpu_ms"] + (["gpu_ms", "speedup"] if CUDA_AVAILABLE else [])].to_string(index=False, float_format="%.3f"))
print()

print("--- dtw_pairwise (batch, seq_length=100) ---")
cols = ["n_sequences", "n_pairs", "cpu_sec"]
if CUDA_AVAILABLE:
    cols += ["gpu_sec", "speedup"]
print(df_pairwise[cols].to_string(index=False, float_format="%.4f"))
print()

print("--- Open boundary overhead (CPU: numpy DP vs tslearn) ---")
print(df_open[["seq_length", "cpu_standard_ms", "cpu_open_end_ms", "open_end_overhead"]].to_string(index=False, float_format="%.3f"))
print()

if CUDA_AVAILABLE:
    peak_speedup_single = df_single["speedup"].max()
    peak_speedup_pairwise = df_pairwise["speedup"].max()
    print(f"Peak GPU speedup (single pair): {peak_speedup_single:.1f}x")
    print(f"Peak GPU speedup (pairwise):    {peak_speedup_pairwise:.1f}x")

In [ ]:
from baleen._cuda_dtw import cleanup
cleanup()